#### Template OPENCV et PYAUDIO

In [ ]:
import cv2
import numpy as np
import pyaudio
import sys

# --- CONFIGURATION PARAMÉTRÉ ---
# Vidéo
CAM_ID = 0          # Index de la caméra (0 = défaut)
WIDTH = 640         # Largeur de capture
HEIGHT = 480        # Hauteur de capture

# Audio
CHUNK = 1024        # Taille du buffer (plus petit = moins de latence, plus de CPU)
FORMAT = pyaudio.paInt16
CHANNELS = 1        # Mono
RATE = 44100        # Fréquence d'échantillonnage (Standard CD)

# --- INITIALISATION ---
# Initialisation de la caméra
cap = cv2.VideoCapture(CAM_ID)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, WIDTH)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, HEIGHT)

# Initialisation de l'audio
p = pyaudio.PyAudio()
try:
    stream = p.open(format=FORMAT, channels=CHANNELS, rate=RATE, 
                    input=True, frames_per_buffer=CHUNK)
except Exception as e:
    print(f"Erreur Audio : {e}")
    sys.exit()

print("Système prêt. Appuyez sur 'q' ou cliquez sur 'STOP' pour quitter.")

# --- FONCTION DE CALLBACK BOUTON ---
def on_stop_button(event, x, y, flags, param):
    """Gère le clic sur la zone du bouton STOP"""
    if event == cv2.EVENT_LBUTTONDOWN:
        if 10 <= x <= 110 and 10 <= y <= 50:
            param['running'] = False

state = {'running': True}
cv2.namedWindow('Interface IA Temps Réel')
cv2.setMouseCallback('Interface IA Temps Réel', on_stop_button, state)

while state['running']:
    # 1. ACQUISITION
    ret, frame = cap.read()
    if not ret: break
    
    try:
        # exception_on_overflow=False évite le crash si le CPU ralentit
        audio_data = stream.read(CHUNK, exception_on_overflow=False)
        audio_np = np.frombuffer(audio_data, dtype=np.int16)
    except:
        audio_np = np.zeros(CHUNK)

    # ---------------------------------------------------------
    # 2. SECTION TRAITEMENT VIDÉO (Ex: Classification, YOLO, etc.)
    # ---------------------------------------------------------
    # C'est ici qu'on appellerait model.predict(frame)
    # Pour l'exemple, on passe juste en gris
    processed_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    processed_frame = cv2.cvtColor(processed_frame, cv2.COLOR_GRAY2BGR) # Repasse en BGR pour dessiner

    # ---------------------------------------------------------
    # 3. SECTION TRAITEMENT AUDIO (Ex: FFT, Classification sonore)
    # ---------------------------------------------------------
    # Calcul du volume RMS pour l'exemple
    volume = np.sqrt(np.mean(audio_np.astype(np.float32)**2))
    
    # ---------------------------------------------------------
    # 4. SECTION AFFICHAGE ET UI
    # ---------------------------------------------------------
    
    # A. TEXTE
    cv2.putText(frame, f"Volume: {int(volume)}", (120, 40), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    # B. BOUTON "STOP" (Rectangle + Texte)
    cv2.rectangle(frame, (10, 10), (110, 50), (0, 0, 255), -1)
    cv2.putText(frame, "STOP", (30, 40), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

    # C. LINEPLOT (Polylines)
    # Création d'un mini-graphe de l'onde audio
    pts_x = np.linspace(10, 200, num=CHUNK) # x de 10 à 200
    # On scale l'audio pour qu'il tienne dans 50px de hauteur
    pts_y = 450 - (audio_np[:CHUNK] / 1000).astype(int) 
    
    # On stack X et Y pour créer les points [x, y]
    pts = np.column_stack((pts_x, pts_y)).astype(np.int32)
    cv2.polylines(frame, [pts.reshape((-1, 1, 2))], False, (255, 255, 0), 1)

    # D. AFFICHAGE FINAL
    cv2.imshow('Interface IA Temps Réel', frame)

    # 5. CONDITIONS DE SORTIE
    if cv2.waitKey(1) & 0xFF == ord('q'):
        state['running'] = False

# --- NETTOYAGE FINAL ---
cap.release()
stream.stop_stream()
stream.close()
p.terminate()
cv2.destroyAllWindows()

Système prêt. Appuyez sur 'q' ou cliquez sur 'STOP' pour quitter.
